## Pruebas con Vega-Altair

In [ ]:
import pandas as pd
data = pd.DataFrame({'a': list('CCCDDDEEE'),
                     'b': [2, 7, 4, 1, 2, 6, 8, 4, 7]})


In [ ]:
import altair as alt
chart = alt.Chart(data)

In [ ]:
alt.Chart(data).mark_point()

alt.Chart(...)

In [ ]:
alt.Chart(data).mark_point().encode(
    x='a',
)

alt.Chart(...)

In [ ]:
alt.Chart(data).mark_point().encode(
    x='a',
    y='b'
)

alt.Chart(...)

In [ ]:
alt.Chart(data).mark_point().encode(
    x='a',
    y='average(b)'
)

alt.Chart(...)

In [ ]:
alt.Chart(data).mark_bar().encode(
    y='a',
    x='average(b)'
)

alt.Chart(...)

In [ ]:
import pandas as pd

iris = pd.read_csv('https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv')

In [ ]:
import altair as alt

chart = alt.Chart(iris).mark_point().encode(
    x='sepal_length',
    y='petal_length',
    shape='species',
    color='species',
    tooltip=['sepal_length', 'petal_length', 'species']
)

chart

alt.Chart(...)

In [1]:
import pandas as pd

meteo = pd.read_csv('https://datos.madrid.es/egob/catalogo/300754-12751538-meteorologia-tiempo-real-acumula.csv', sep=';')   

meteo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 56 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PROVINCIA       2900 non-null   int64  
 1   MUNICIPIO       2900 non-null   int64  
 2   ESTACION        2900 non-null   int64  
 3   MAGNITUD        2900 non-null   int64  
 4   PUNTO_MUESTREO  0 non-null      float64
 5   ANO             2900 non-null   int64  
 6   MES             2900 non-null   int64  
 7   DIA             2900 non-null   int64  
 8   H01             2900 non-null   float64
 9   V01             2900 non-null   object 
 10  H02             2900 non-null   float64
 11  V02             2900 non-null   object 
 12  H03             2900 non-null   float64
 13  V03             2900 non-null   object 
 14  H04             2900 non-null   float64
 15  V04             2900 non-null   object 
 16  H05             2900 non-null   float64
 17  V05             2900 non-null   o

In [2]:
meteo['date'] = pd.to_datetime(
    meteo[['ANO','MES','DIA']].rename(columns={'ANO':'year','MES':'month','DIA':'day'}),
    errors='coerce'
)

In [4]:
# Pivotar pares HNN (valor) y VNN (validez) a formato largo
import re
# Detectar columnas HNN y VNN
h_cols = sorted([c for c in meteo.columns if re.match(r'^H\d{2}$', c)], key=lambda x: int(x[1:]))
v_cols = sorted([c for c in meteo.columns if re.match(r'^V\d{2}$', c)], key=lambda x: int(x[1:]))
# Asegurar correspondencia entre HNN y VNN
hours = []
for n in range(1, 25):
    h = f'H{n:02d}'
    v = f'V{n:02d}'
    if h in meteo.columns and v in meteo.columns:
        hours.append(n)
# id_vars: columnas a mantener (todas excepto H/V)
id_vars = [c for c in meteo.columns if not re.match(r'^[HV]\d{2}$', c)]
# Construir lista de dataframes por hora y concatenar
parts = []
for n in hours:
    hcol = f'H{n:02d}'
    vcol = f'V{n:02d}'
    df = meteo[id_vars + [hcol, vcol]].copy()
    df = df.rename(columns={hcol: 'value', vcol: 'validity'})
    df['hour'] = n
    parts.append(df)
meteo_long = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=id_vars + ['value','validity','hour'])
# Convertir tipos
meteo_long['value'] = pd.to_numeric(meteo_long['value'], errors='coerce')
# Normalizar validez si procede (ej. eliminar espacios)
meteo_long['validity'] = meteo_long['validity'].astype(str).str.strip().replace({'nan': None})
# Si existe la columna `date`, construir `datetime` combinando fecha + hora
if 'date' in meteo_long.columns:
    meteo_long['hour0_23'] = meteo_long['hour'] % 24
    meteo_long['datetime'] = pd.to_datetime(meteo_long['date']) + pd.to_timedelta(meteo_long['hour0_23'], unit='h')
    # Si hour==24, sumar 1 día a la marca temporal (H24 = 24:00 -> siguiente día 00:00)
    meteo_long.loc[meteo_long['hour'] == 24, 'datetime'] += pd.Timedelta(days=1)
# Reordenar columnas: id_vars + hour + datetime + value + validity
cols_order = id_vars + ['hour'] + (['datetime'] if 'datetime' in meteo_long.columns else []) + ['value','validity']
meteo_long = meteo_long.loc[:, [c for c in cols_order if c in meteo_long.columns]]
# Mostrar ejemplo
meteo_long.shape

(69600, 13)

In [5]:
import pandas as pd
import re

# Asegura índice único para wide_to_long
meteo = meteo.reset_index(drop=True)
meteo['record_id'] = meteo.index

# Aplicar wide_to_long a los stubs H y V (H01..H24, V01..V24)
wtl = pd.wide_to_long(
    meteo,
    stubnames=['H', 'V'],
    i='record_id',
    j='hour',
    sep='',
    suffix='\\d{2}'
).reset_index()

# Renombrar y convertir tipos
wtl = wtl.rename(columns={'H': 'value', 'V': 'validity'})
wtl['hour'] = pd.to_numeric(wtl['hour'], errors='coerce').astype('Int64')
wtl['value'] = pd.to_numeric(wtl['value'], errors='coerce')
wtl['validity'] = wtl['validity'].astype(str).str.strip().replace({'nan': None})

# Construir datetime si existe `date` (mapear H24 -> siguiente día 00:00)
if 'date' in wtl.columns:
    wtl['hour0_23'] = wtl['hour'] % 24
    wtl['datetime'] = pd.to_datetime(wtl['date']) + pd.to_timedelta(wtl['hour0_23'], unit='h')
    wtl.loc[wtl['hour'] == 24, 'datetime'] += pd.Timedelta(days=1)

# Orden final (ajusta id_vars si quieres mantener/seleccionar otros campos)
cols_keep = [c for c in meteo.columns if c not in ('record_id',)+tuple([f'H{n:02d}' for n in range(1,25)]+[f'V{n:02d}' for n in range(1,25)])]
result = wtl[cols_keep + ['hour', 'datetime', 'value', 'validity']]

In [7]:
# Filtrar solo registros con MAGNITUD == 83
# Asegúrate de que las columnas existan en 'result' y filtrar por MAGNITUD==83 y validity=='V'
if 'MAGNITUD' in result.columns:
    mask = result['MAGNITUD'] == 83
    if 'validity' in result.columns:
        mask &= result['validity'].fillna('').str.upper() == 'V'
    result_83 = result[mask].reset_index(drop=True)
else:
    # Si no existe, crear vacío con las mismas columnas
    result_83 = result.loc[0:0].iloc[0:0].copy()
result_83.shape
result_83.head()

,PROVINCIA,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,DIA,date,hour,datetime,value,validity
0,28,79,102,83,NaN,2025,11,13,2025-11-13,1,2025-11-13 01:00:00,11.5,V
1,28,79,102,83,NaN,2025,11,14,2025-11-14,1,2025-11-14 01:00:00,9.4,V
2,28,79,102,83,NaN,2025,11,15,2025-11-15,1,2025-11-15 01:00:00,6.3,V
3,28,79,102,83,NaN,2025,11,16,2025-11-16,1,2025-11-16 01:00:00,6.1,V
4,28,79,102,83,NaN,2025,11,17,2025-11-17,1,2025-11-17 01:00:00,5.2,V


In [13]:
# Celda duplicada: Gráfico Altair (x=datetime, y=value), color y forma por ESTACION
import altair as alt
alt.data_transformers.enable("vegafusion")
# Asegurar tipos
result_83['datetime'] = pd.to_datetime(result_83['datetime'], errors='coerce')
result_83['value'] = pd.to_numeric(result_83['value'], errors='coerce')
# Elegir columna de estación
station_col = 'ESTACION' if 'ESTACION' in result_83.columns else None
plot_df = result_83.dropna(subset=['datetime', 'value']).copy()
if plot_df.empty:
    print('No hay datos con datetime y value válidos para graficar')
else:
    if station_col:
        line = alt.Chart(plot_df).mark_line().encode(
            x=alt.X('datetime:T', title='Datetime'),
            y=alt.Y('value:Q', title='Value'),
            color=alt.Color(f'{station_col}:N', title='Estación')
        )
        points = alt.Chart(plot_df).mark_point(filled=True, size=40).encode(
            x='datetime:T',
            y='value:Q',
            color=alt.Color(f'{station_col}:N'),
            shape=alt.Shape(f'{station_col}:N', title='Estación')
        )
        chart = (line + points).properties(width=800, height=350).interactive()
    else:
        chart = alt.Chart(plot_df).mark_line().encode(
            x='datetime:T',
            y='value:Q'
        ).properties(width=800, height=350).interactive()
chart

alt.LayerChart(...)